In [1]:
!pip install datasets scikit-learn -q

In [2]:
import pandas as pd
import numpy as np
import re
from datasets import load_dataset
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score

# ── Data Loading ──────────────────────────────────────────────────────────────
# AutoTherm indoor dataset from HuggingFace
dataset = load_dataset("kopetri/AutoTherm", "indoor")
df = dataset["train"].to_pandas()

# Extract participant ID from filename
def extract_participant_id(filename):
    match = re.search(r"participant_\d+", filename)
    return match.group() if match else "unknown"

df["participant_id"] = df["file_name"].apply(extract_participant_id)

# ── Subject-wise split ────────────────────────────────────────────────────────
# Same split used in every experiment throughout this project
# Participants 14, 16, 20 held out as test set
# Participant 16 selected because they report all 7 thermal comfort classes
TEST_PARTICIPANTS  = ["participant_14", "participant_16", "participant_20"]
TRAIN_PARTICIPANTS = [p for p in df["participant_id"].unique()
                      if p not in TEST_PARTICIPANTS]

train_df = df[df["participant_id"].isin(TRAIN_PARTICIPANTS)].copy()
test_df  = df[df["participant_id"].isin(TEST_PARTICIPANTS)].copy()

print(f"Training participants: {sorted(TRAIN_PARTICIPANTS)}")
print(f"Test participants:     {TEST_PARTICIPANTS}")
print(f"Train shape: {train_df.shape}")
print(f"Test shape:  {test_df.shape}")
print(f"\nTrain label distribution:")
print(train_df["Label"].value_counts().sort_index())

README.md:   0%|          | 0.00/8.57k [00:00<?, ?B/s]

indoor/train-00000-of-00002.parquet: reconstructing file:   0%|          |  0.00B / 29.8MB            

indoor/train-00000-of-00002.parquet: downloading bytes:           |  0.00B            

indoor/train-00001-of-00002.parquet: reconstructing file:   0%|          |  0.00B / 30.0MB            

indoor/train-00001-of-00002.parquet: downloading bytes:           |  0.00B            

indoor/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 7.41MB            

indoor/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1566728 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/194829 [00:00<?, ? examples/s]

Training participants: ['participant_10', 'participant_11', 'participant_15', 'participant_17', 'participant_18', 'participant_19', 'participant_2', 'participant_21', 'participant_3', 'participant_4', 'participant_6', 'participant_7', 'participant_8']
Test participants:     ['participant_14', 'participant_16', 'participant_20']
Train shape: (1276709, 36)
Test shape:  (290019, 36)

Train label distribution:
Label
-3     48020
-2    116840
-1    263290
 0    307137
 1    178333
 2    202958
 3    160131
Name: count, dtype: int64


In [3]:
# ── SHARED UTILITY FUNCTIONS ──────────────────────────────────────────────────
# BUG-1 FIX: In wk1-wk6, LabelEncoder was fitted independently on each
# dataframe passed to prepare_features(). This meant train and test could
# receive different integer assignments for the same categorical value.
#
# THE FIX: fit the encoder ONCE on training data only, then apply that
# same fitted encoder to both train and test. This is the standard
# scikit-learn pattern and what wk7 already does correctly.

# Features to drop — includes all label-derived columns to prevent leakage
DROP_COLS = [
    "file_name", "Timestamp", "participant_id",
    "Label", "Label_3class",
    # Zero-variance features identified in wk1 exploration
    "Air-Velocity", "Metabolic-Rate",
    # Pose keypoints — out of scope for this project
    "Nose", "Neck", "RShoulder", "RElbow", "LShoulder",
    "LElbow", "REye", "LEye", "REar", "LEar",
    # Emotion columns — out of scope
    "Emotion-Self", "Emotion-ML",
]

def add_3class_label(df):
    """Add collapsed 3-class label column.

    Mapping: Cold (≤-2) → 0, Neutral (-1 to +1) → 1, Warm (≥+2) → 2
    This mapping is consistent across all experiments in this project.
    """
    df = df.copy()
    df["Label_3class"] = df["Label"].apply(
        lambda x: 0 if x <= -2 else (2 if x >= 2 else 1)
    )
    return df


def fit_label_encoders(train_df, drop_cols=DROP_COLS):
    """Fit LabelEncoders on training data only.

    Returns a dict of {column_name: fitted_LabelEncoder} for all
    categorical columns in the training feature matrix.

    This is the BUG-1 fix: encoders are fitted ONCE on training data
    and stored for reuse on test data, ensuring consistent encoding.

    Args:
        train_df: Training DataFrame (before dropping label columns)
        drop_cols: Columns to exclude from the feature matrix

    Returns:
        Dict mapping column names to fitted LabelEncoder objects
    """
    X_train = train_df.drop(
        columns=[c for c in drop_cols if c in train_df.columns]
    )
    encoders = {}
    for col in X_train.select_dtypes(include=["object", "category"]).columns:
        le = LabelEncoder()
        le.fit(X_train[col].astype(str))
        encoders[col] = le
    return encoders


def prepare_features(df, target_col, encoders, drop_cols=DROP_COLS):
    """Prepare feature matrix and target vector for one split.

    Applies pre-fitted encoders to categorical columns — never refits.
    Unseen categories default to -1 to avoid crashes on test data.

    Args:
        df: DataFrame for this split (train or test)
        target_col: Name of the target column ('Label' or 'Label_3class')
        encoders: Dict of fitted LabelEncoders from fit_label_encoders()
        drop_cols: Columns to exclude from feature matrix

    Returns:
        Tuple of (X, y) as numpy arrays
    """
    df = df.copy()
    y = df[target_col].values

    X = df.drop(
        columns=[c for c in drop_cols if c in df.columns]
    )

    # Apply pre-fitted encoders — never refit on test data
    for col, le in encoders.items():
        if col in X.columns:
            X[col] = X[col].astype(str).map(
                lambda val, le=le: (
                    le.transform([val])[0]
                    if val in le.classes_ else -1
                )
            )

    # Convert datetime columns safely (pandas 2.0 compatible)
    for col in X.select_dtypes(include=["datetime64"]).columns:
        X[col] = X[col].astype(np.int64) // 10**9

    X = X.fillna(X.median(numeric_only=True))

    return X.values, y


print("Shared utility functions defined.")
print(f"DROP_COLS ({len(DROP_COLS)} columns): {DROP_COLS}")

Shared utility functions defined.
DROP_COLS (19 columns): ['file_name', 'Timestamp', 'participant_id', 'Label', 'Label_3class', 'Air-Velocity', 'Metabolic-Rate', 'Nose', 'Neck', 'RShoulder', 'RElbow', 'LShoulder', 'LElbow', 'REye', 'LEye', 'REar', 'LEar', 'Emotion-Self', 'Emotion-ML']


In [4]:
# ── FEATURE PREPARATION ───────────────────────────────────────────────────────
# Step 1: Add 3-class labels to both splits
train_df = add_3class_label(train_df)
test_df  = add_3class_label(test_df)

# Step 2: Fit encoders on training data ONLY
# This is the BUG-1 fix — encoders never see test data during fitting
print("Fitting LabelEncoders on training data only...")
encoders = fit_label_encoders(train_df)
print(f"Encoders fitted for {len(encoders)} categorical columns:")
for col, le in encoders.items():
    print(f"  {col}: {list(le.classes_)}")

# Step 3: Prepare 7-class features
print("\nPreparing 7-class feature matrices...")
X_train_7, y_train_7 = prepare_features(train_df, "Label", encoders)
X_test_7,  y_test_7  = prepare_features(test_df,  "Label", encoders)

print(f"X_train_7 shape: {X_train_7.shape}")
print(f"X_test_7 shape:  {X_test_7.shape}")

# Step 4: Prepare 3-class features
print("\nPreparing 3-class feature matrices...")
X_train_3, y_train_3 = prepare_features(train_df, "Label_3class", encoders)
X_test_3,  y_test_3  = prepare_features(test_df,  "Label_3class", encoders)

print(f"X_train_3 shape: {X_train_3.shape}")
print(f"X_test_3 shape:  {X_test_3.shape}")

# Verify no label columns leaked into features
print("\nFeature columns (should not contain Label or Label_3class):")
feature_cols = [c for c in train_df.columns
                if c not in DROP_COLS and
                c not in ["Label", "Label_3class"]]
print(f"Number of features: {len(feature_cols)}")

Fitting LabelEncoders on training data only...
Encoders fitted for 1 categorical columns:
  Gender: ['Female', 'Male']

Preparing 7-class feature matrices...
X_train_7 shape: (1276709, 18)
X_test_7 shape:  (290019, 18)

Preparing 3-class feature matrices...
X_train_3 shape: (1276709, 18)
X_test_3 shape:  (290019, 18)

Feature columns (should not contain Label or Label_3class):
Number of features: 18


In [5]:
# ── BASELINE EXPERIMENT — 7-CLASS ─────────────────────────────────────────────
# Training a Random Forest with the same hyperparameters used throughout
# this project (n_estimators=100, random_state=42, n_jobs=-1)
# This replicates the baseline from wk1 using the corrected encoder

print("Training 7-class baseline Random Forest...")
print("(This takes 2-3 minutes)")

rf_7 = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)
rf_7.fit(X_train_7, y_train_7)

y_pred_7 = rf_7.predict(X_test_7)

macro_f1_7 = f1_score(y_test_7, y_pred_7, average="macro", zero_division=0)
print(f"\n7-class Macro F1 (BUG-1 fixed): {macro_f1_7:.4f}")
print(f"7-class Macro F1 (original wk1): 0.2858")
print(f"Difference: {macro_f1_7 - 0.2858:+.4f}")

print("\nClassification Report:")
print(classification_report(
    y_test_7, y_pred_7,
    zero_division=0
))

Training 7-class baseline Random Forest...
(This takes 2-3 minutes)

7-class Macro F1 (BUG-1 fixed): 0.2858
7-class Macro F1 (original wk1): 0.2858
Difference: +0.0000

Classification Report:
              precision    recall  f1-score   support

          -3       0.00      0.00      0.00     22556
          -2       0.16      0.26      0.20     19285
          -1       0.64      0.70      0.67     85260
           0       0.16      0.22      0.18     39044
           1       0.13      0.31      0.18     16824
           2       0.36      0.35      0.35     51786
           3       0.78      0.28      0.41     55264

    accuracy                           0.39    290019
   macro avg       0.32      0.30      0.29    290019
weighted avg       0.44      0.39      0.39    290019



In [6]:
# ── BASELINE EXPERIMENT — 3-CLASS ─────────────────────────────────────────────
print("Training 3-class baseline Random Forest...")

rf_3 = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)
rf_3.fit(X_train_3, y_train_3)

y_pred_3 = rf_3.predict(X_test_3)

macro_f1_3 = f1_score(y_test_3, y_pred_3, average="macro", zero_division=0)
print(f"\n3-class Macro F1 (BUG-1 fixed): {macro_f1_3:.4f}")
print(f"3-class Macro F1 (original wk1): 0.7163")
print(f"Difference: {macro_f1_3 - 0.7163:+.4f}")

print("\nClassification Report:")
print(classification_report(
    y_test_3, y_pred_3,
    zero_division=0
))

# ── VERIFICATION SUMMARY ──────────────────────────────────────────────────────
print("\n" + "="*60)
print("BUG-1 FIX VERIFICATION SUMMARY")
print("="*60)
print(f"7-class Macro F1 — Original: 0.2858 | Fixed: {macro_f1_7:.4f} | "
      f"Change: {macro_f1_7 - 0.2858:+.4f}")
print(f"3-class Macro F1 — Original: 0.7163 | Fixed: {macro_f1_3:.4f} | "
      f"Change: {macro_f1_3 - 0.7163:+.4f}")
print("="*60)
print("\nConclusion: BUG-1 fix does not change baseline results.")
print("Core findings are valid. Gender encoding was consistent")
print("between train and test in the original implementation")
print("because Female/Male appeared in both splits in the same order.")

Training 3-class baseline Random Forest...

3-class Macro F1 (BUG-1 fixed): 0.7163
3-class Macro F1 (original wk1): 0.7163
Difference: -0.0000

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.55      0.71     41841
           1       0.66      0.97      0.78    141128
           2       0.92      0.51      0.65    107050

    accuracy                           0.74    290019
   macro avg       0.86      0.68      0.72    290019
weighted avg       0.80      0.74      0.72    290019


BUG-1 FIX VERIFICATION SUMMARY
7-class Macro F1 — Original: 0.2858 | Fixed: 0.2858 | Change: +0.0000
3-class Macro F1 — Original: 0.7163 | Fixed: 0.7163 | Change: -0.0000

Conclusion: BUG-1 fix does not change baseline results.
Core findings are valid. Gender encoding was consistent
between train and test in the original implementation
because Female/Male appeared in both splits in the same order.
